#  Phase 1: Data Collection & Preprocessing
**DSA 210 — Spring 2026 | Zeynep Altundal | 31978 | Sabancı University**

---

##  Notebook Overview
This notebook covers **data collection, loading, and preprocessing** for the Accident Risk Prediction project.

**Output:** Clean 440K-record DataFrame ready for EDA and modeling.

### Environment Setup
Installing required libraries:
-  Kaggle API client for dataset download
-  statsmodels statistical tests (proportion z-test)

In [ ]:
!pip install kaggle statsmodels -q

In [ ]:
import os, json

kaggle_creds = {
    "username": "#######",
    "key": "#######"
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)
!chmod 600 /root/.kaggle/kaggle.json
print("Done")

Done


In [ ]:
!kaggle datasets download -d sobhanmoosavi/us-accidents
!unzip -q us-accidents.zip
!ls *.csv

Dataset URL: https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents
License(s): CC-BY-NC-SA-4.0
100% 653M/653M [00:03<00:00, 217MB/s]

US_Accidents_March23.csv


###  Library Imports & Configuration
Loading all required libraries for data manipulation, visualization, and statistical testing.
Setting global plot style and creating the `figures/` directory for saving outputs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
import warnings
import os

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
os.makedirs("figures", exist_ok=True)
print(" Libraries loaded!")

 Libraries loaded!


###  Data Loading & Sampling
The full dataset contains **7,728,394 records** across 46 features.
We take a reproducible random sample of **500,000 records** (random_state=42) —
statistically representative while keeping computation feasible on Colab.

In [ ]:
print("Loading dataset... (may take ~1 min)")
df_full = pd.read_csv("US_Accidents_March23.csv", low_memory=False)
print(f"Full dataset: {df_full.shape}")

df = df_full.sample(n=500_000, random_state=42).copy()
del df_full
print(f" Working sample: {df.shape}")
df.head(3)

Loading dataset... (may take ~1 min)
Full dataset: (7728394, 46)
 Working sample: (500000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
7133276,A-7182628,Source1,1,2020-04-17 09:29:30,2020-04-17 10:29:30,26.706900,-80.119360,26.706900,-80.119360,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
5363845,A-5404588,Source1,2,2022-04-21 10:01:00.000000000,2022-04-21 11:44:08.000000000,38.781024,-121.265820,38.780377,-121.265815,0.045,...,False,False,True,False,False,False,Day,Day,Day,Day
155993,A-156000,Source3,3,2016-08-12 16:45:00,2016-08-12 17:15:00,33.985249,-84.269348,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day


### Preprocessing
Cleaning the data and engineering new features for analysis.

| Step | Description |
|------|-------------|
| Timestamp parsing | Extract Hour, DayOfWeek, Month, Year, IsWeekend |
| Duration | Compute accident duration in minutes |
| TimeOfDay | Morning (6-12) / Afternoon (12-17) / Evening (17-21) / Night |
| HighRisk label | Binary: Severity ≥ 3 → 1, else → 0 |
| Missing values | Drop rows missing Severity, Hour, Weather, Temperature |
| Temperature | Remove physically impossible values (< -50°F or > 130°F) |
| Weather | Map 100+ raw conditions → 8 broad categories |

In [ ]:
# Parse timestamps
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["End_Time"]   = pd.to_datetime(df["End_Time"],   errors="coerce")

# Temporal features
df["Hour"]      = df["Start_Time"].dt.hour
df["DayOfWeek"] = df["Start_Time"].dt.day_name()
df["Month"]     = df["Start_Time"].dt.month
df["Year"]      = df["Start_Time"].dt.year
df["IsWeekend"] = df["Start_Time"].dt.dayofweek >= 5

# Duration
df["Duration_min"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60

# Time of day buckets
def time_of_day(hour):
    if 6 <= hour < 12:   return "Morning"
    elif 12 <= hour < 17: return "Afternoon"
    elif 17 <= hour < 21: return "Evening"
    else:                 return "Night"

df["TimeOfDay"] = df["Hour"].apply(time_of_day)

# High risk binary label (Severity 3 or 4)
df["HighRisk"] = (df["Severity"] >= 3).astype(int)

# Drop rows missing key columns
key_cols = ["Severity", "Hour", "Start_Time", "Weather_Condition", "Temperature(F)"]
before = len(df)
df.dropna(subset=key_cols, inplace=True)
df = df[(df["Temperature(F)"] > -50) & (df["Temperature(F)"] < 130)]
print(f"Dropped {before - len(df):,} rows → Final: {len(df):,} rows")

# Simplify weather conditions
weather_map = {
    "Clear": ["Clear", "Fair"],
    "Cloudy": ["Cloudy", "Overcast", "Mostly Cloudy", "Partly Cloudy", "Scattered Clouds"],
    "Rain": ["Rain", "Light Rain", "Heavy Rain", "Drizzle", "Showers", "Light Drizzle"],
    "Snow": ["Snow", "Light Snow", "Heavy Snow", "Blowing Snow", "Sleet"],
    "Fog":  ["Fog", "Haze", "Mist", "Smoke", "Patches of Fog"],
    "Storm":["Thunderstorm", "Thunder", "Squalls"],
    "Wind": ["Windy", "Breezy", "Blowing Dust"],
}

def map_weather(cond):
    if pd.isna(cond): return "Other"
    for cat, kws in weather_map.items():
        if any(k.lower() in str(cond).lower() for k in kws):
            return cat
    return "Other"

df["WeatherCategory"] = df["Weather_Condition"].apply(map_weather)

print(f"\nSeverity distribution:\n{df['Severity'].value_counts().sort_index()}")
print(f"\nHigh-risk rate: {df['HighRisk'].mean()*100:.1f}%")
print(" Preprocessing done!")

Dropped 59,682 rows → Final: 440,318 rows

Severity distribution:
Severity
1      4319
2    342267
3     82262
4     11470
Name: count, dtype: int64

High-risk rate: 21.3%
 Preprocessing done!


###  Preprocessing Results
- **Dropped:** 59,682 rows with missing or invalid values (11.9% of sample)
- **Final sample:** 440,318 records ready for analysis
- **Severity breakdown:** Severity 1 → 1% | 2 → 78% | 3 → 19% | 4 → 2.6%
- **High-risk rate:** 21.3% of accidents are Severity 3 or 4
- **Note:** End_Lat/End_Lng has 44% missing values dataset-wide — not used in this phase
- **Note:** Precipitation(in) has 28% missing — will be handled in Phase 3